# Plataforma de análisis de tráfico vehicular con visión artificial
**Proyecto 14 · Big Data / Ingeniería de Software + IA** · Notebook de evaluación y registro

Este notebook **no es un experimento aislado**: analiza lo que la plataforma ya procesó y lo guarda otra vez en MongoDB.

```
Video ──► Backend FastAPI ──► Servicio IA (YOLO26 + ByteTrack | baseline OpenCV)
                │                         │
                └──────► MongoDB ◄────────┘      (sources, jobs, counts)
                            │
                  ESTE NOTEBOOK  (evaluación contra el conteo manual)
                            │
                            ▼
        MongoDB: ground_truth, model_versions, evaluations   ──►  futuro frontend / dashboard
```

**Qué hace**
1. Lee desde MongoDB las fuentes, los trabajos procesados y los conteos por intervalo.
2. Registra la **verdad de terreno** (conteo manual) y la **versión de cada modelo**.
3. Compara **realidad vs baseline vs YOLO26** (error de conteo, error por categoría, FPS).
4. Revisa el flujo en el tiempo y la **consistencia** entre los trabajos y la colección de series temporales.
5. Muestra fotogramas con las detecciones de YOLO26 y la línea de conteo (y, opcionalmente, un video anotado).
6. Mide una **disponibilidad básica** de la API.
7. Guarda la evaluación completa en MongoDB (`evaluations`) y exporta CSV y gráficas a `notebooks/resultados/`.

## 0. Antes de ejecutar

Desde **Anaconda Prompt**, con el entorno del proyecto:

```
conda activate trafico
pip install -r notebooks\requirements-notebook.txt
docker compose up mongo -d
jupyter notebook
```

- Para las secciones 1 a 5 y 8 solo hace falta **MongoDB encendido**.
- La sección 6 (disponibilidad) necesita el backend y el servicio de IA corriendo.
- La sección 7 (fotogramas con YOLO26) necesita `ultralytics`, el modelo `models/yolo26n.pt` y los videos en `data/uploads`.
- Ejecuta las celdas **en orden** (`Shift + Enter`). Puedes repetir el notebook cuando quieras: cada ejecución añade un nuevo documento en `evaluations`.

In [ ]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display
from pymongo import MongoClient

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
plt.rcParams["figure.dpi"] = 110

NOTEBOOK_VERSION = "1.0"
MONGO_URI = "mongodb://localhost:27017"
MONGO_DB = "trafico"
API_URL = "http://localhost:8000"


def encontrar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar docker-compose.yml (raíz del proyecto)."""
    p = Path(inicio or Path.cwd()).resolve()
    for d in [p, *p.parents]:
        if (d / "docker-compose.yml").exists():
            return d
    return p


REPO_ROOT = encontrar_raiz()
RESULTS_DIR = REPO_ROOT / "notebooks" / "resultados"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Se reutiliza la misma lógica de conteo del servicio de IA (coherencia con el sistema)
sys.path.insert(0, str(REPO_ROOT / "ai-service"))
try:
    from app.counting import LineCounter, line_to_pixels
except Exception as e:
    LineCounter = line_to_pixels = None
    print("No se pudo importar ai-service/app/counting.py:", e)

CATEGORIES = ["car", "motorcycle", "bus", "truck"]
CAT_ES = {"car": "Carros", "motorcycle": "Motos", "bus": "Buses", "truck": "Camiones"}
METHODS = ["baseline", "yolo26"]
METHOD_LABEL = {"baseline": "Baseline (OpenCV MOG2)", "yolo26": "YOLO26n + ByteTrack"}
METHOD_COLOR = {"baseline": "#d08770", "yolo26": "#5e81ac"}

print("Raíz del proyecto :", REPO_ROOT)
print("Resultados en     :", RESULTS_DIR)

## 1. Conexión a MongoDB

In [ ]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
client.admin.command("ping")          # falla si MongoDB no está encendido
db = client[MONGO_DB]
print("Conectado a MongoDB, base de datos:", MONGO_DB)
print("Colecciones y número de documentos:")
for nombre in sorted(db.list_collection_names()):
    print(f"  {nombre:<15} {db[nombre].count_documents({})}")

## 2. Datos registrados por la plataforma
### 2.1 Fuentes de video y trabajos procesados

In [ ]:
fuentes = list(db.sources.find({}))
df_fuentes = pd.DataFrame(
    [{"source_id": str(s["_id"]), "nombre": s.get("name"), "línea [x1,y1,x2,y2]": s.get("line"),
      "autorizada": s.get("authorized"), "creada_por": s.get("created_by")} for s in fuentes],
    columns=["source_id", "nombre", "línea [x1,y1,x2,y2]", "autorizada", "creada_por"])
print("Fuentes de video:")
display(df_fuentes)

trabajos = list(db.jobs.find({"status": "completado"}))


def fila_trabajo(j):
    meta = j.get("ai_meta") or {}
    return {"job_id": str(j["_id"]), "video": j.get("original_name"), "método": j.get("method"),
            "fuente": j.get("source_id"), "FPS": meta.get("processing_fps"), "seg": meta.get("elapsed_s"),
            "fotogramas": meta.get("frames_processed"), "modelo": meta.get("model"),
            "totales": j.get("totals"), "creado": j.get("created_at")}


df_trabajos = pd.DataFrame([fila_trabajo(j) for j in trabajos])
print(f"Trabajos completados: {len(trabajos)}")
display(df_trabajos)

### 2.2 Trazabilidad: versiones de modelo usadas
Cada trabajo guardó el modelo, la versión de la librería, el dispositivo y los parámetros. Aquí se registran como **versiones de modelo** (`model_versions`).

In [ ]:
def info_version(j):
    meta = j.get("ai_meta") or {}
    lib, ver = ("ultralytics", meta.get("ultralytics_version")) if meta.get("method") == "yolo26" \
        else ("opencv", meta.get("opencv_version"))
    return {"method": meta.get("method"), "model": meta.get("model"), "library": lib,
            "library_version": ver, "device": meta.get("device"),
            "service_version": meta.get("service_version")}


versiones = {}
for j in trabajos:
    v = info_version(j)
    versiones[(v["method"], v["model"], v["library"], v["library_version"])] = v

ahora = datetime.now(timezone.utc)
for clave, v in versiones.items():
    filtro = {"method": v["method"], "model": v["model"], "library": v["library"],
              "library_version": v["library_version"]}
    db.model_versions.update_one(
        filtro,
        {"$set": {"device": v["device"], "service_version": v["service_version"], "last_seen": ahora},
         "$setOnInsert": {"first_seen": ahora}},
        upsert=True)

display(pd.DataFrame(list(versiones.values())))
print("Versiones registradas en la colección model_versions:", len(versiones))

## 3. Verdad de terreno (conteo manual)
Conteo hecho **a mano** viendo cada video. Criterio: se cuenta un vehículo cuando su parte delantera pasa por la línea de conteo (a la mitad de la altura del video).

> Para sumar más videos: procésalos con la plataforma (`yolo26` y `baseline`), cuéntalos a mano y agrega una entrada aquí, usando el **nombre exacto del archivo**.

In [ ]:
GROUND_TRUTH = {
    "VideoPrueba-ParaBigdata1.mp4": {"car": 16, "motorcycle": 8, "bus": 3, "truck": 3},
    "VideoPrueba-ParaBigdata2.mp4": {"car": 16, "motorcycle": 7, "bus": 0, "truck": 1},
}
NOTAS_GT = {
    "VideoPrueba-ParaBigdata1.mp4": "Carros: puede haber incertidumbre sobre cuántos cruzaron exactamente la línea.",
    "VideoPrueba-ParaBigdata2.mp4": "",
}
CRITERIO_GT = "Conteo manual; se cuenta el vehículo cuando su parte delantera cruza la línea de conteo."


def ultimo_trabajo(lista, video, metodo):
    candidatos = [j for j in lista if j.get("original_name") == video and j.get("method") == metodo]
    return max(candidatos, key=lambda j: j["created_at"]) if candidatos else None


JOBS_USADOS = {}
faltantes = []
for video in GROUND_TRUTH:
    for metodo in METHODS:
        j = ultimo_trabajo(trabajos, video, metodo)
        if j is None:
            faltantes.append((video, metodo))
        else:
            JOBS_USADOS[(video, metodo)] = j

if faltantes:
    print("Atención: faltan trabajos completados para:")
    for v, m in faltantes:
        print("  -", v, "·", m)
ignorados = sorted({j.get("original_name") for j in trabajos} - set(GROUND_TRUTH))
if ignorados:
    print("Videos procesados sin verdad de terreno (se ignoran en la evaluación):", ignorados)

ahora = datetime.now(timezone.utc)
for video, conteo in GROUND_TRUTH.items():
    db.ground_truth.update_one(
        {"video_name": video},
        {"$set": {"video_name": video, "counts": conteo, "criterio": CRITERIO_GT,
                  "nota": NOTAS_GT.get(video, ""), "updated_at": ahora}},
        upsert=True)
print("Verdad de terreno guardada en MongoDB (ground_truth):", list(GROUND_TRUTH))
print("Trabajos usados en la evaluación:")
for (v, m), j in JOBS_USADOS.items():
    print(f"  {v:<32} {m:<9} job {j['_id']}")

## 4. Evaluación: realidad vs baseline vs YOLO26
### 4.1 Conteo por categoría

In [ ]:
filas = []
for (video, metodo), j in JOBS_USADOS.items():
    totales = j.get("totals") or {}
    for cat in CATEGORIES:
        real = GROUND_TRUTH[video][cat]
        pred = int(totales.get(cat, 0))
        filas.append({"video": video, "method": metodo, "category": cat, "real": real,
                      "predicho": pred, "error": pred - real, "error_abs": abs(pred - real)})
df_eval = pd.DataFrame(filas)
assert not df_eval.empty, "No hay trabajos para evaluar: procesa los videos con la plataforma y revisa la sección 3."

for video in GROUND_TRUTH:
    sub = df_eval[df_eval.video == video]
    if sub.empty:
        continue
    tabla = sub.pivot_table(index="category", columns="method", values="predicho", aggfunc="first")
    tabla = tabla.reindex(CATEGORIES)
    tabla.insert(0, "real", pd.Series(GROUND_TRUTH[video]))
    tabla.loc["TOTAL"] = tabla.sum()
    tabla.index = [CAT_ES.get(i, i) for i in tabla.index]
    print(video)
    display(tabla)

### 4.2 Métricas por video y globales
- **error_conteo_total_%**: (predicho − real) / real sobre el total de vehículos. Puede engañar, porque los errores de distintas categorías se compensan.
- **error_abs_categoria_%**: suma de los errores absolutos por categoría / total real. Es la medida más honesta.
- **MAE_por_categoria**: error absoluto medio por categoría (en vehículos).
- **FPS**: fotogramas procesados por segundo en CPU.

In [ ]:
fps_map = {(v, m): (j.get("ai_meta") or {}).get("processing_fps") for (v, m), j in JOBS_USADOS.items()}

resumen = (df_eval.groupby(["video", "method"], as_index=False)
           .agg(real_total=("real", "sum"), predicho_total=("predicho", "sum"), error_abs_suma=("error_abs", "sum")))
resumen["error_conteo_total_%"] = ((resumen.predicho_total - resumen.real_total) / resumen.real_total * 100).round(1)
resumen["error_abs_categoria_%"] = (resumen.error_abs_suma / resumen.real_total * 100).round(1)
resumen["MAE_por_categoria"] = (resumen.error_abs_suma / len(CATEGORIES)).round(2)
resumen["FPS"] = [fps_map.get((v, m)) for v, m in zip(resumen.video, resumen.method)]
print("Por video y método:")
display(resumen)

global_ = (resumen.groupby("method", as_index=False)
           .agg(real_total=("real_total", "sum"), predicho_total=("predicho_total", "sum"),
                error_abs_suma=("error_abs_suma", "sum"), FPS_promedio=("FPS", "mean")))
global_["error_abs_categoria_%"] = (global_.error_abs_suma / global_.real_total * 100).round(1)
global_["FPS_promedio"] = global_.FPS_promedio.round(1)
print("Globales (todos los videos):")
display(global_)

### 4.3 Gráficas

In [ ]:
videos = [v for v in GROUND_TRUTH if not df_eval[df_eval.video == v].empty]
corto = lambda v: v.replace("VideoPrueba-", "").replace(".mp4", "")

# 1) Conteo por categoría
fig, axes = plt.subplots(1, len(videos), figsize=(6.3 * len(videos), 4.3), squeeze=False)
x = np.arange(len(CATEGORIES)); w = 0.27
for ax, video in zip(axes[0], videos):
    sub = df_eval[df_eval.video == video]
    ax.bar(x - w, [GROUND_TRUTH[video][c] for c in CATEGORIES], w, label="Real", color="#4c566a")
    for k, m in enumerate(METHODS):
        vals = [int(sub[(sub.method == m) & (sub.category == c)].predicho.sum()) for c in CATEGORIES]
        ax.bar(x + k * w, vals, w, label=METHOD_LABEL[m], color=METHOD_COLOR[m])
    ax.set_xticks(x); ax.set_xticklabels([CAT_ES[c] for c in CATEGORIES])
    ax.set_title(corto(video)); ax.set_ylabel("Vehículos"); ax.grid(axis="y", alpha=.3)
axes[0][0].legend(fontsize=8)
fig.suptitle("Conteo por categoría: realidad vs baseline vs YOLO26", y=1.02)
plt.savefig(RESULTS_DIR / "conteo_por_categoria.png", bbox_inches="tight"); plt.show()

# 2) Error absoluto por categoría (%) y 3) velocidad
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
xv = np.arange(len(videos))
for k, m in enumerate(METHODS):
    vals = [float(resumen[(resumen.video == v) & (resumen.method == m)]["error_abs_categoria_%"].iloc[0])
            if not resumen[(resumen.video == v) & (resumen.method == m)].empty else 0 for v in videos]
    barras = a1.bar(xv + (k - .5) * .35, vals, .35, label=METHOD_LABEL[m], color=METHOD_COLOR[m])
    a1.bar_label(barras, fmt="%.0f%%", fontsize=8)
a1.set_xticks(xv); a1.set_xticklabels([corto(v) for v in videos])
a1.set_ylabel("Error absoluto por categoría (%)"); a1.set_title("Error de conteo (menor es mejor)")
a1.legend(fontsize=8); a1.grid(axis="y", alpha=.3)

fps_vals = [float(global_[global_.method == m].FPS_promedio.iloc[0]) if (global_.method == m).any() else 0 for m in METHODS]
barras = a2.bar([METHOD_LABEL[m] for m in METHODS], fps_vals, color=[METHOD_COLOR[m] for m in METHODS])
a2.bar_label(barras, fmt="%.1f", fontsize=9)
a2.set_ylabel("FPS (CPU)"); a2.set_title("Velocidad de procesamiento"); a2.grid(axis="y", alpha=.3)
plt.setp(a2.get_xticklabels(), fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "error_y_velocidad.png", bbox_inches="tight"); plt.show()

## 5. Flujo en el tiempo (colección de series temporales `counts`)
Cada vehículo contado se guardó en `counts` con su marca de tiempo (`recorded_at` + segundo del video). Aquí se reconstruye el flujo por intervalo y se comprueba que **coincide** con los totales de cada trabajo (consistencia entre servicio de IA, API y base de datos).

In [ ]:
def cargar_conteos(job):
    docs = list(db.counts.find({"meta.job_id": str(job["_id"])}))
    base = job["recorded_at"].replace(tzinfo=None)
    filas = [{"intervalo_s": int(round((d["ts"].replace(tzinfo=None) - base).total_seconds())),
              "category": d["meta"]["category"], "direction": d["meta"]["direction"], "count": d["count"]}
             for d in docs]
    return pd.DataFrame(filas, columns=["intervalo_s", "category", "direction", "count"])


SERIES = {k: cargar_conteos(j) for k, j in JOBS_USADOS.items()}

print("Chequeo de consistencia: totales del trabajo vs documentos de la colección counts")
todo_ok = True
for (video, metodo), j in JOBS_USADOS.items():
    df = SERIES[(video, metodo)]
    desde_mongo = {k: int(v) for k, v in df.groupby("category")["count"].sum().items()} if not df.empty else {}
    del_trabajo = {k: int(v) for k, v in (j.get("totals") or {}).items() if v}
    ok = desde_mongo == del_trabajo
    todo_ok &= ok
    print(f"  {corto(video):<14} {metodo:<9} {'OK' if ok else 'DIFERENCIA'}  counts={desde_mongo}  trabajo={del_trabajo}")
print("Resultado:", "todo consistente" if todo_ok else "hay diferencias, revisar")

In [ ]:
fig, axes = plt.subplots(1, len(videos), figsize=(6 * len(videos), 3.8), squeeze=False)
for ax, video in zip(axes[0], videos):
    for m in METHODS:
        df = SERIES.get((video, m))
        if df is None or df.empty:
            continue
        s = df.groupby("intervalo_s")["count"].sum()
        ax.plot(s.index, s.values, marker="o", label=METHOD_LABEL[m], color=METHOD_COLOR[m])
    ax.set_title(corto(video)); ax.set_xlabel("Segundo del video (inicio del intervalo)")
    ax.set_ylabel("Vehículos contados"); ax.grid(alpha=.3)
axes[0][0].legend(fontsize=8)
fig.suptitle("Flujo vehicular por intervalo", y=1.02)
plt.savefig(RESULTS_DIR / "flujo_por_intervalo.png", bbox_inches="tight"); plt.show()

# MAE del flujo agregado (opcional): requiere conteo manual por intervalo
# Formato: {"VideoPrueba-ParaBigdata1.mp4": {0: 12, 10: 11, 20: 7}}  (vehículos por intervalo de 10 s)
GROUND_TRUTH_INTERVALOS = {}

if not GROUND_TRUTH_INTERVALOS:
    print("MAE del flujo agregado: pendiente. Para calcularlo, completa GROUND_TRUTH_INTERVALOS "
          "con tu conteo manual por intervalo de 10 s.")
else:
    filas = []
    for video, real_int in GROUND_TRUTH_INTERVALOS.items():
        for m in METHODS:
            df = SERIES.get((video, m))
            if df is None:
                continue
            pred = df.groupby("intervalo_s")["count"].sum().to_dict()
            errores = [abs(pred.get(t, 0) - n) for t, n in real_int.items()]
            filas.append({"video": video, "method": m, "MAE_flujo": round(float(np.mean(errores)), 2)})
    display(pd.DataFrame(filas))

## 6. Verificación visual con YOLO26
Para **ver** lo que detecta el modelo y comprobar dónde está la línea de conteo. La línea y los parámetros salen del propio trabajo guardado en MongoDB. Requiere `ultralytics`, el modelo y los videos en este computador.

In [ ]:
import cv2

COCO = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
COLOR_BGR = {"car": (172, 129, 94), "motorcycle": (0, 165, 255), "bus": (80, 175, 76), "truck": (200, 90, 200)}
_MODELO = {}


def cargar_modelo():
    if "m" not in _MODELO:
        from ultralytics import YOLO
        ruta = REPO_ROOT / "models" / "yolo26n.pt"
        _MODELO["m"] = YOLO(str(ruta) if ruta.exists() else "yolo26n.pt")
    return _MODELO["m"]


def dibujar_linea(img, linea, grosor=3):
    h, w = img.shape[:2]
    (x1, y1), (x2, y2) = line_to_pixels(linea, w, h)
    cv2.line(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), grosor)


def vista_previa(job, fracciones=(0.25, 0.5, 0.75)):
    ruta = job.get("video_path")
    if not ruta or not Path(ruta).exists():
        print("Video no encontrado en este computador:", ruta)
        return
    p = job["ai_meta"]["params"]
    modelo = cargar_modelo()
    cap = cv2.VideoCapture(ruta)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fig, axes = plt.subplots(1, len(fracciones), figsize=(4.2 * len(fracciones), 7.6), squeeze=False)
    for ax, f in zip(axes[0], fracciones):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * f))
        ok, frame = cap.read()
        if not ok:
            continue
        r = modelo.predict(frame, classes=list(COCO), conf=p["conf"], imgsz=p["imgsz"], device="cpu", verbose=False)[0]
        for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.int().tolist(), r.boxes.conf.tolist()):
            cat = COCO[c]
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), COLOR_BGR[cat], 2)
            cv2.putText(frame, f"{cat} {s:.2f}", (int(x1), max(int(y1) - 5, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_BGR[cat], 1, cv2.LINE_AA)
        dibujar_linea(frame, p["line"])
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); ax.axis("off")
        ax.set_title(f"Fotograma al {int(f * 100)} %")
    cap.release()
    plt.tight_layout(); plt.show()


for (video, metodo), j in JOBS_USADOS.items():
    if metodo != "yolo26":
        continue
    print(video)
    try:
        vista_previa(j)
    except ImportError:
        print("  ultralytics no está instalado en este entorno; se omite la vista previa.")
    except Exception as e:
        print("  No se pudo generar la vista previa:", e)

In [ ]:
# Video anotado (opcional): cajas, números de seguimiento, línea de conteo y contador en vivo.
# Tarda aproximadamente lo mismo que procesar el video (1 a 2 minutos por video en CPU).
RENDER_VIDEOS = False
MAX_SEGUNDOS = None      # por ejemplo 15 para renderizar solo el inicio


def renderizar_video(job, salida, max_segundos=None):
    p = job["ai_meta"]["params"]
    ruta = job["video_path"]
    cap = cv2.VideoCapture(ruta); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0; cap.release()
    modelo = cargar_modelo()
    escritor = None
    contador = None
    flujo = modelo.track(source=ruta, stream=True, persist=True, tracker="bytetrack.yaml",
                         classes=list(COCO), conf=p["conf"], imgsz=p["imgsz"], vid_stride=p["stride"],
                         device="cpu", verbose=False)
    for i, r in enumerate(flujo):
        t = i * p["stride"] / fps
        if max_segundos and t > max_segundos:
            break
        frame = r.orig_img.copy()
        h, w = r.orig_shape
        if contador is None:
            a, b = line_to_pixels(p["line"], w, h)
            contador = LineCounter(a, b, p["interval_seconds"])
            escritor = cv2.VideoWriter(str(salida), cv2.VideoWriter_fourcc(*"mp4v"), fps / p["stride"], (w, h))
        if r.boxes is not None and r.boxes.id is not None:
            for tid, c, (x1, y1, x2, y2) in zip(r.boxes.id.int().tolist(), r.boxes.cls.int().tolist(), r.boxes.xyxy.tolist()):
                cat = COCO[c]
                contador.update(tid, cat, ((x1 + x2) / 2, (y1 + y2) / 2), t)
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), COLOR_BGR[cat], 2)
                cv2.putText(frame, f"id{tid} {cat}", (int(x1), max(int(y1) - 5, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_BGR[cat], 1, cv2.LINE_AA)
        dibujar_linea(frame, p["line"])
        tot = {}
        for (_, cat, _), n in contador.agg.items():
            tot[cat] = tot.get(cat, 0) + n
        cv2.rectangle(frame, (0, 0), (190, 22 * (len(CATEGORIES) + 1)), (0, 0, 0), -1)
        for k, cat in enumerate(CATEGORIES):
            cv2.putText(frame, f"{CAT_ES[cat]}: {tot.get(cat, 0)}", (6, 20 + 22 * k),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
        escritor.write(frame)
    if escritor is not None:
        escritor.release()
    return {c: sum(n for (_, cat, _), n in contador.agg.items() if cat == c) for c in CATEGORIES} if contador else {}


if RENDER_VIDEOS:
    for (video, metodo), j in JOBS_USADOS.items():
        if metodo != "yolo26":
            continue
        salida = RESULTS_DIR / f"anotado_{Path(video).stem}.mp4"
        try:
            tot = renderizar_video(j, salida, MAX_SEGUNDOS)
            print(f"{video}: video anotado en {salida}\n  conteo del render: {tot}\n  conteo guardado:   {j.get('totals')}")
        except Exception as e:
            print(f"{video}: no se pudo renderizar ({e})")
else:
    print("RENDER_VIDEOS = False: cámbialo a True para generar los videos anotados en notebooks/resultados/.")

## 7. Disponibilidad y tiempo de respuesta básicos
Prueba sencilla sobre `GET /health` de la API (que a su vez comprueba MongoDB y el servicio de IA). Si el backend no está encendido, el resultado será 0 %.

In [ ]:
def probar_disponibilidad(n=20, pausa=0.2):
    ok, latencias = 0, []
    for _ in range(n):
        t0 = time.perf_counter()
        try:
            r = requests.get(f"{API_URL}/health", timeout=6)
            if r.status_code == 200 and all(v == "ok" for v in r.json().values()):
                ok += 1
            latencias.append((time.perf_counter() - t0) * 1000)
        except Exception:
            pass
        time.sleep(pausa)
    return {"solicitudes": n, "correctas": ok, "disponibilidad_%": round(ok / n * 100, 1),
            "latencia_media_ms": round(float(np.mean(latencias)), 1) if latencias else None,
            "latencia_p95_ms": round(float(np.percentile(latencias, 95)), 1) if latencias else None}


disponibilidad = probar_disponibilidad()
display(pd.DataFrame([disponibilidad]))

## 8. Registro en MongoDB y exportación
Se guarda un documento en la colección `evaluations` con todo lo calculado (métricas por video, método y categoría, versiones de modelo, trabajos usados y disponibilidad). Los CSV y las gráficas se exportan a `notebooks/resultados/` para el informe.

In [ ]:
def a_python(df):
    """DataFrame -> lista de dicts con tipos nativos (MongoDB no acepta tipos de NumPy)."""
    return json.loads(df.to_json(orient="records"))


evaluacion = {
    "created_at": datetime.now(timezone.utc),
    "notebook_version": NOTEBOOK_VERSION,
    "videos": list(GROUND_TRUTH),
    "criterio_verdad_terreno": CRITERIO_GT,
    "por_video_metodo": a_python(resumen),
    "por_categoria": a_python(df_eval),
    "global_por_metodo": a_python(global_),
    "jobs_usados": {f"{v}|{m}": str(j["_id"]) for (v, m), j in JOBS_USADOS.items()},
    "versiones_modelo": list(versiones.values()),
    "disponibilidad": disponibilidad,
    "consistencia_counts_vs_jobs": bool(todo_ok),
    "metricas_pendientes": ["mAP (requiere fotogramas anotados con cajas)",
                            "MAE del flujo agregado (requiere conteo manual por intervalo)"],
}
res = db.evaluations.insert_one(evaluacion)
print("Evaluación guardada en MongoDB, id:", res.inserted_id)

df_eval.to_csv(RESULTS_DIR / "evaluacion_por_categoria.csv", index=False, encoding="utf-8-sig")
resumen.to_csv(RESULTS_DIR / "resumen_por_video_metodo.csv", index=False, encoding="utf-8-sig")
global_.to_csv(RESULTS_DIR / "resumen_global.csv", index=False, encoding="utf-8-sig")
print("Archivos exportados en", RESULTS_DIR)
for f in sorted(RESULTS_DIR.iterdir()):
    print("  ", f.name)

print("\nÚltimas evaluaciones registradas:")
ultimas = sorted(db.evaluations.find({}), key=lambda d: d["created_at"], reverse=True)[:3]
display(pd.DataFrame([{"id": str(d["_id"]), "fecha": d["created_at"], "versión_notebook": d.get("notebook_version"),
                       "videos": len(d.get("videos", []))} for d in ultimas]))
print("Colecciones finales:", {n: db[n].count_documents({}) for n in sorted(db.list_collection_names())})

## 9. Conclusiones
Las cifras de abajo se calculan con los datos de esta ejecución.

In [ ]:
g = global_.set_index("method")
if {"baseline", "yolo26"} <= set(g.index):
    e_b, e_y = g.loc["baseline", "error_abs_categoria_%"], g.loc["yolo26", "error_abs_categoria_%"]
    f_b, f_y = g.loc["baseline", "FPS_promedio"], g.loc["yolo26", "FPS_promedio"]
    print(f"- Videos evaluados: {len(videos)}  |  vehículos reales contados a mano: {int(g.loc['yolo26', 'real_total'])}")
    print(f"- Error absoluto por categoría: baseline {e_b}%  vs  YOLO26 {e_y}%.")
    mejor = "YOLO26" if e_y < e_b else "el baseline"
    print(f"- {mejor[0].upper() + mejor[1:]} comete menos error de conteo en esta muestra.")
    if f_y:
        print(f"- Velocidad en CPU: baseline {f_b} FPS vs YOLO26 {f_y} FPS (el baseline es ~{f_b / f_y:.1f}x más rápido).")
    print("- Categorías con más error en YOLO26:")
    peores = (df_eval[df_eval.method == "yolo26"].groupby("category")["error_abs"].sum()
              .sort_values(ascending=False))
    for cat, err in peores.items():
        print(f"    {CAT_ES[cat]}: {int(err)} vehículos de error acumulado")
else:
    print("Faltan resultados de alguno de los dos métodos para concluir.")

### Limitaciones y trabajo futuro
- **Muestra pequeña**: pocos videos cortos y conteos manuales hechos por una persona; los porcentajes son indicativos, no estadística definitiva.
- **Motos**: pequeñas y rápidas; el baseline las infla (fragmenta manchas grandes) y YOLO26 tiende a no contarlas. Probar `stride = 1` e `imgsz = 640`.
- **Baseline**: clasifica solo por el área de la mancha y no distingue buses.
- **Mismo ángulo y línea**: los resultados dependen de la posición de la línea de conteo; otro ángulo exige una fuente nueva.
- **Métricas pendientes**: mAP (requiere fotogramas anotados con cajas) y MAE del flujo agregado (requiere conteo manual por intervalo).
- **Siguiente paso técnico**: ajuste fino de YOLO26 con fotogramas etiquetados de estos videos (entrenamiento por épocas) y comparación antes/después, registrando cada entrenamiento en `model_versions`.
- **Frontend**: el dashboard leerá `jobs`, `counts` y `evaluations` desde la API.